# YOLO Annotation QA Pipeline

Measures how much bad labels hurt a YOLO model and how much of that a cleaning step recovers.

Before running: in the notebook settings turn **Accelerator = GPU** and **Internet = On**
(Internet is needed to clone the repo and download the dataset and YOLO weights).

In [ ]:
# Setup
!git clone https://github.com/podiavaibhav/annotation-qa-yolo.git
%cd annotation-qa-yolo
!pip install -q -r requirements.txt

In [ ]:
# Download the dataset and write a yaml the pipeline can use
!python main.py prepare --dataset african-wildlife.yaml

from pathlib import Path
DATA = Path('outputs/prepared_dataset.txt').read_text().strip()
print('Using', DATA)

Optional but recommended: a 1-epoch smoke run first. It checks that everything runs end to end
(its numbers are meaningless) and costs a few minutes.

In [ ]:
!python main.py --config configs/smoke.yaml run-all --data "$DATA"

Full run: 3 training seeds for each of Original / Noisy / Cleaned, plus the out-of-fold models.
If the session dies, rerun this cell with `--resume` added so finished runs are reused.

In [ ]:
!python main.py run-all --data "$DATA"

In [ ]:
# Results
import pandas as pd
from IPython.display import display, Image, Markdown

display(pd.read_csv('outputs/experiments/comparison.csv'))
display(Markdown(Path('outputs/experiments/final_report.md').read_text()))
display(Markdown(Path('outputs/experiments/detector_eval.md').read_text()))
display(Image('outputs/experiments/performance_comparison.png'))

In [ ]:
# Bundle the results (without model weights) so they can be downloaded from the Output tab
!zip -qr results.zip outputs/experiments outputs/analysis outputs/cleaning outputs/model_audit/*.csv outputs/noisy_dataset/corruption_manifest.json -x "*/weights/*" "*.pt"